### Get the pretrained models

In [1]:
import torch

In [2]:
import torchvision

vit_weights = torchvision.models.ViT_B_16_Weights.DEFAULT
vit_transforms = vit_weights.transforms()

resnet_weights = torchvision.models.ResNet152_Weights.DEFAULT
resnet_transforms = resnet_weights.transforms()

### Get the data

In [3]:
from torchvision import datasets
from pathlib import Path

vit_data_dir = Path("vit_data")

vit_train_transforms = torchvision.transforms.Compose([
    torchvision.transforms.TrivialAugmentWide(),
    vit_transforms 
])

# Getting ViT traning data
vit_train_data = datasets.FGVCAircraft(root=vit_data_dir,
                                       split="train",
                                       transform=vit_train_transforms,
                                       download=True)

# Get ViT test data
vit_test_data = datasets.FGVCAircraft(root=vit_data_dir,
                                      split="test",
                                      transform=vit_transforms,
                                      download=True)

In [4]:
resnet_data_dir = Path("resnet_data")


resnet_train_transforms = torchvision.transforms.Compose([
    torchvision.transforms.TrivialAugmentWide(),
    resnet_transforms
])

# Getting ViT traning data
resnet_train_data = datasets.FGVCAircraft(root=resnet_data_dir,
                                          split="train",
                                          transform=resnet_train_transforms,
                                          download=True)

resnet_test_data = datasets.FGVCAircraft(root=resnet_data_dir,
                                          split="test",
                                          transform=resnet_transforms,
                                          download=True) 

### Create the dataloaders for the data 

In [13]:
from torch.utils.data import DataLoader
import os

NUM_WORKERS = os.cpu_count()
BATCH_SIZE = 64

vit_train_dataloder = DataLoader(dataset=vit_train_data,
                                 batch_size=BATCH_SIZE,
                                 shuffle=True,
                                 num_workers=NUM_WORKERS,
                                 pin_memory=True)

vit_test_dataloder = DataLoader(dataset=vit_test_data,
                                 batch_size=BATCH_SIZE,
                                 shuffle=True,
                                 num_workers=NUM_WORKERS,
                                 pin_memory=True)

resnet_train_dataloder = DataLoader(dataset=resnet_train_data,
                                 batch_size=BATCH_SIZE,
                                 shuffle=True,
                                 num_workers=NUM_WORKERS,
                                 pin_memory=True)

resnet_test_dataloder = DataLoader(dataset=resnet_test_data,
                                 batch_size=BATCH_SIZE,
                                 shuffle=True,
                                 num_workers=NUM_WORKERS,
                                 pin_memory=True)

In [8]:
num_classes = len(resnet_train_data.classes)
num_classes

100

### Create functions to get the weights, models, and transforms

In [9]:
import torch

device = "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [10]:
from torch import nn

def create_vit(device,
               num_classes: int = 100,
               seed: int = 42):
    weights = torchvision.models.ViT_B_16_Weights.DEFAULT
    model = torchvision.models.vit_b_16(weights=weights).to(device)

    # Freeze feature extraction layers
    for param in model.parameters():
        param.requires_grad = False

    torch.manual_seed(seed)
    torch.mps.manual_seed(seed)
    # Create a new classifier layer
    
    model.heads = nn.Sequential(
        nn.Linear(in_features=768, out_features=num_classes)
    )

    return model

In [11]:
from torchinfo import summary

# Create an example model
vit_example = create_vit(device=device,
                         num_classes=num_classes,
                         seed=42)

# Get a summary of the model
vit_example

VisionTransformer(
  (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  (encoder): Encoder(
    (dropout): Dropout(p=0.0, inplace=False)
    (layers): Sequential(
      (encoder_layer_0): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (dropout): Dropout(p=0.0, inplace=False)
        (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
          (4): Dropout(p=0.0, inplace=False)
        )
      )
      (encoder_layer_1): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_a

In [12]:
def create_resnet(device,
                  num_classes: int = 100,
                  seed: int = 42):
    # Get the model and weights
    weights = torchvision.models.ResNet152_Weights.DEFAULT
    model = torchvision.models.resnet152(weights=weights).to(device)

    # Freeze feature extraction layers
    for param in model.parameters():
        param.requires_grad = False

    torch.manual_seed(seed)
    torch.mps.manual_seed(seed)

    # Create a new classifier layer
    model.fc = nn.Linear(in_features=2048, out_features=num_classes)

    return model

In [13]:
resnet_example = create_resnet(device=device,
                               num_classes=num_classes,
                               seed=42)

resnet_example

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

### Create a loop to train both models for 3 epochs and write the resutls to a file

In [14]:
model_names = ["vit", "resnet"]

In [15]:
from torch.utils.tensorboard import SummaryWriter
import engine

EPOCHS = 3

for model_name in model_names:
    if model_name == "vit":
        vit = create_vit(device=device,
                         num_classes=num_classes,
                         seed=42)
        vit.to(device)
        writer = SummaryWriter(os.path.join("results/3_epochs", model_name))

        optimizer = torch.optim.Adam(vit.parameters(),
                                     lr=1e-3)
        loss_fn = nn.CrossEntropyLoss()
        
        vit_results = engine.train(model=vit,
                                   train_dataloader=vit_train_dataloder,
                                   test_dataloader=vit_test_dataloder,
                                   optimizer=optimizer,
                                   writer=writer,
                                   loss_fn=loss_fn,
                                   epochs=EPOCHS,
                                   device=device)
        
    else:
        resnet = create_resnet(device=device)

        writer = SummaryWriter(os.path.join("results/3_epochs", model_name))

        optimizer = torch.optim.Adam(params=resnet.parameters(),
                                     lr=1e-3)
        
        loss_fn = nn.CrossEntropyLoss()

        resnet_results = engine.train(model=vit,
                                      train_dataloader=resnet_train_dataloder,
                                      test_dataloader=resnet_test_dataloder,
                                      optimizer=optimizer,
                                      loss_fn=loss_fn,
                                      writer=writer,
                                      epochs=EPOCHS,
                                      device=device)

  0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
# # resnet = create_resnet(device=device,
# #                        num_classes=num_classes,
# #                        seed = 42)
# ImageNumber: int = 5

# image = resnet_test_dataloder.dataset[ImageNumber][0].unsqueeze(dim=0).to(device)

# resnet.to(device)
# resnet.eval()
# with torch.inference_mode():
#     target_image_pred = resnet(image)

#     target_image_pred_probs = torch.softmax(target_image_pred, dim=1) 

#     target_image_pred_label = torch.argmax(target_image_pred, dim=1)
    
#     plt.figure()
#     plt.imshow(image.squeeze().permute(1,2,0).cpu())
#     plt.title(f"Pred: {resnet_test_data.classes[target_image_pred_label]} | Prob: {target_image_pred_probs.max():.3f}")
#     plt.axis(False)
#     print(f"Actual Label: {resnet_test_data.classes[resnet_test_dataloder.dataset[ImageNumber][1]]}")

In [15]:
# %%time

# # Time the ViT model
# ImageNumber: int = 5

# image = resnet_test_dataloder.dataset[ImageNumber][0].unsqueeze(dim=0).to(device)

# resnet.to(device)
# resnet.eval()
# with torch.inference_mode():
#     target_image_pred = resnet(image)

#     target_image_pred_probs = torch.softmax(target_image_pred, dim=1) 

#     target_image_pred_label = torch.argmax(target_image_pred, dim=1)

In [16]:
# %%time

# # Time the ViT model
# ImageNumber: int = 5

# image = resnet_test_dataloder.dataset[ImageNumber][0].unsqueeze(dim=0).to(device)

# vit.to(device)
# vit.eval()

# with torch.inference_mode():
#     target_image_pred = vit(image)

#     target_image_pred_probs = torch.softmax(target_image_pred, dim=1) 

#     target_image_pred_label = torch.argmax(target_image_pred, dim=1)

In [17]:
# import matplotlib.pyplot as plt

# plt.figure()
# plt.imshow(image.squeeze().permute(1,2,0).cpu())
# plt.title(f"Pred: {resnet_test_data.classes[target_image_pred_label]} | Prob: {target_image_pred_probs.max():.3f}")
# plt.axis(False)
# print(f"Actual Label: {resnet_test_data.classes[resnet_test_dataloder.dataset[ImageNumber][1]]}")

In [18]:
# import pandas as pd
# resnet_df = pd.DataFrame(resnet_results)
# resnet_df

In [19]:
# vit_df = pd.DataFrame(vit_results)
# vit_df

### Train the ViT model for 10 epochs

In [13]:
class_names = vit_test_data.classes
num_classes = len(vit_test_data.classes)

In [14]:
import torch
from tqdm.auto import tqdm

def train(model: torch.nn.Module,
          writer,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader:  torch.utils.data.DataLoader,
          loss_fn: torch.nn.Module,
          optimizer: torch.optim.Optimizer,
          device,
          epochs: int = 10,
          save_model: bool = False,
          save_model_path: str = "./models",
          model_name: str = "vit_model"):
    
    for epoch in tqdm(range(epochs)):
        # Put the model in train mode
        model.to(device)
        model.train()

        # Setup train loss and train accuracy values
        train_loss, train_acc = 0, 0

        for batch, (X, y) in enumerate(train_dataloader):
            # Send data to target device
            X, y = X.to(device), y.to(device)

            # Forward pass
            y_pred = model(X)

            # Calculate and accumulate loss 
            loss = loss_fn(y_pred, y)
            train_loss += loss.item()

            # Optimizer zero grad
            optimizer.zero_grad()

            # Loss backward
            loss.backward()

            # Optimizer step
            optimizer.step()

            # Calculate and accumulate accuray metrics across all batches
            y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
            train_acc += (y_pred_class == y).sum().item()/len(y_pred)

        # Adjust metrics to get average loss and accuracy per batch
        train_loss = train_loss / len(train_dataloader)
        train_acc = train_acc / len(train_dataloader)

        # Put the model in eval mode
        model.to(device)
        model.eval()

        # Setup test loss and test accuracy values
        test_loss, test_acc = 0, 0

        with torch.inference_mode():
            # Loop through DataLoader batches
            for batch, (X, y) in enumerate(test_dataloader):
                # Send data to target deivce
                X, y = X.to(device), y.to(device)

                # Forward pass
                test_pred_logits = model(X)

                # Calculate and accumulate loss
                loss = loss_fn(test_pred_logits, y)
                test_loss += loss.item()

                # Calculate and accumulate accuracy
                test_pred_labels = test_pred_logits.argmax(dim=1) 
                test_acc += ((test_pred_labels == y).sum().item() / len(test_pred_labels))

            # Adjust metrics to get average loss and accuracy per batch
            test_loss = test_loss / len(train_dataloader)
            test_acc = test_acc / len(test_dataloader)
            
            results = {"train_loss": [],
                        "train_acc": [],
                        "test_loss": [],
                        "test_acc": []}
            print(
                    f"Epoch: {epoch+1} | "
                    f"train_loss: {train_loss:.4f} | "
                    f"train_acc: {train_acc:.4f} | "
                    f"test_loss: {test_loss:.4f} | "
                    f"test_acc: {test_acc:.4f}")
            results["train_loss"].append(train_loss)
            results["train_acc"].append(train_acc)
            results["test_loss"].append(test_loss)
            results["test_acc"].append(test_acc)

            writer.add_scalar(tag="Train Loss",
                              scalar_value=train_loss,
                              global_step=epoch)

            writer.add_scalar(tag="Train Accuracy",
                              scalar_value=train_acc,
                              global_step=epoch)             

            writer.add_scalar(tag="Test Loss",
                              scalar_value=test_loss,
                              global_step=epoch) 
            writer.add_scalar(tag="Test Accuracy",
                              scalar_value=test_acc,
                              global_step=epoch) 
            writer.add_graph(model=model,
                             input_to_model=torch.randn(64, 3, 224, 224).to(device))
    if save_model == True:
        print(f"[INFO] Saving {model_name} model to {save_model_path}")
        MODEL_PATH = Path(save_model_path) 
        MODEL_PATH.mkdir(parents=True,
                         exist_ok=True)
        MODEL_SAVE_PATH = MODEL_PATH/model_name
        torch.save(obj=model.state_dict(),
                   f=MODEL_SAVE_PATH)
        return results
    else:
        return results

In [15]:
device = "mps"
device

'mps'

In [23]:
from torch.utils.tensorboard import SummaryWriter

vit_10_epochs = create_vit(device=device,
                           num_classes=num_classes,
                           seed=42)

vit_10_epochs.to(device)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=vit_10_epochs.parameters(),
                             lr=1e-3)
writer = SummaryWriter(log_dir=os.path.join("./results", "10_epochs"))

train(model=vit_10_epochs,
      train_dataloader=vit_train_dataloder,
      writer=writer,
      test_dataloader=vit_test_dataloder,
      loss_fn=loss_fn,
      optimizer=optimizer,
      device=device,
      epochs=10,
      save_model=True,
      save_model_path="./models",
      model_name="vit_10_epochs")

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 4.2990 | train_acc: 0.0580 | test_loss: 3.9562 | test_acc: 0.1143


/opt/homebrew/Caskroom/miniconda/base/envs/AircraftDetection/lib/python3.10/site-packages/torch/__init__.py:1209: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert condition, message


Epoch: 2 | train_loss: 3.8070 | train_acc: 0.1569 | test_loss: 3.6965 | test_acc: 0.1608
Epoch: 3 | train_loss: 3.5522 | train_acc: 0.2278 | test_loss: 3.5327 | test_acc: 0.1948
Epoch: 4 | train_loss: 3.3427 | train_acc: 0.2891 | test_loss: 3.3977 | test_acc: 0.2293
Epoch: 5 | train_loss: 3.1716 | train_acc: 0.3244 | test_loss: 3.2722 | test_acc: 0.2555
Epoch: 6 | train_loss: 3.0463 | train_acc: 0.3553 | test_loss: 3.1745 | test_acc: 0.2953
Epoch: 7 | train_loss: 2.8987 | train_acc: 0.3977 | test_loss: 3.1061 | test_acc: 0.3106
Epoch: 8 | train_loss: 2.8021 | train_acc: 0.4284 | test_loss: 3.0520 | test_acc: 0.3190
Epoch: 9 | train_loss: 2.7126 | train_acc: 0.4365 | test_loss: 2.9885 | test_acc: 0.3271
Epoch: 10 | train_loss: 2.6322 | train_acc: 0.4500 | test_loss: 2.9121 | test_acc: 0.3424
[INFO] Saving vit_10_epochs model to ./models


{'train_loss': [2.6321668849801116],
 'train_acc': [0.4499803459119497],
 'test_loss': [2.912063202768002],
 'test_acc': [0.3423938679245283]}

In [25]:
from torch.utils.tensorboard import SummaryWriter

vit_20_epochs = create_vit(device=device,
                           num_classes=num_classes,
                           seed=42)

vit_20_epochs.to(device)

twenty_epochs_loss_fn = torch.nn.CrossEntropyLoss()

twenty_epochs_optimizer = torch.optim.Adam(params=vit_20_epochs.parameters(),
                             lr=1e-3)

writer = SummaryWriter(log_dir=os.path.join("./results", "20_epochs"))

train(model=vit_20_epochs,
      train_dataloader=vit_train_dataloder,
      writer=writer,
      test_dataloader=vit_test_dataloder,
      loss_fn=twenty_epochs_loss_fn,
      optimizer=twenty_epochs_optimizer,
      device=device,
      epochs=20,
      save_model=True,
      save_model_path="./models",
      model_name="vit_20_epochs")

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 4.2990 | train_acc: 0.0580 | test_loss: 3.9562 | test_acc: 0.1143


/opt/homebrew/Caskroom/miniconda/base/envs/AircraftDetection/lib/python3.10/site-packages/torch/__init__.py:1209: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert condition, message


Epoch: 2 | train_loss: 3.8070 | train_acc: 0.1569 | test_loss: 3.6965 | test_acc: 0.1608
Epoch: 3 | train_loss: 3.5522 | train_acc: 0.2278 | test_loss: 3.5327 | test_acc: 0.1948
Epoch: 4 | train_loss: 3.3427 | train_acc: 0.2891 | test_loss: 3.3977 | test_acc: 0.2293
Epoch: 5 | train_loss: 3.1716 | train_acc: 0.3244 | test_loss: 3.2722 | test_acc: 0.2555
Epoch: 6 | train_loss: 3.0463 | train_acc: 0.3553 | test_loss: 3.1745 | test_acc: 0.2953
Epoch: 7 | train_loss: 2.8987 | train_acc: 0.3977 | test_loss: 3.1061 | test_acc: 0.3106
Epoch: 8 | train_loss: 2.8021 | train_acc: 0.4284 | test_loss: 3.0520 | test_acc: 0.3190
Epoch: 9 | train_loss: 2.7126 | train_acc: 0.4365 | test_loss: 2.9885 | test_acc: 0.3271
Epoch: 10 | train_loss: 2.6322 | train_acc: 0.4500 | test_loss: 2.9121 | test_acc: 0.3424
Epoch: 11 | train_loss: 2.5100 | train_acc: 0.4900 | test_loss: 2.8665 | test_acc: 0.3445
Epoch: 12 | train_loss: 2.4340 | train_acc: 0.5179 | test_loss: 2.8421 | test_acc: 0.3375
Epoch: 13 | train_

{'train_loss': [2.0277975932607113],
 'train_acc': [0.58687106918239],
 'test_loss': [2.5330031716598653],
 'test_acc': [0.4001179245283019]}

In [26]:
from torch.utils.tensorboard import SummaryWriter

vit_50_epochs = create_vit(device=device,
                           num_classes=num_classes,
                           seed=42)

vit_50_epochs.to(device)

twenty_epochs_loss_fn = torch.nn.CrossEntropyLoss()

twenty_epochs_optimizer = torch.optim.Adam(params=vit_50_epochs.parameters(),
                             lr=0.01)

writer = SummaryWriter(log_dir=os.path.join("./results", "50_epochs"))

train(model=vit_50_epochs,
      train_dataloader=vit_train_dataloder,
      writer=writer
      test_dataloader=vit_test_dataloder,
      loss_fn=twenty_epochs_loss_fn,
      optimizer=twenty_epochs_optimizer,
      device=device,
      epochs=50,
      save_model=True,
      save_model_path="./models",
      model_name="vit_50_epochs")

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 4.2990 | train_acc: 0.0580 | test_loss: 3.9562 | test_acc: 0.1143
Epoch: 2 | train_loss: 3.8070 | train_acc: 0.1569 | test_loss: 3.6965 | test_acc: 0.1608
Epoch: 3 | train_loss: 3.5522 | train_acc: 0.2278 | test_loss: 3.5327 | test_acc: 0.1948
Epoch: 4 | train_loss: 3.3427 | train_acc: 0.2891 | test_loss: 3.3977 | test_acc: 0.2293
Epoch: 5 | train_loss: 3.1716 | train_acc: 0.3244 | test_loss: 3.2722 | test_acc: 0.2555
Epoch: 6 | train_loss: 3.0463 | train_acc: 0.3553 | test_loss: 3.1745 | test_acc: 0.2953
Epoch: 7 | train_loss: 2.8987 | train_acc: 0.3977 | test_loss: 3.1061 | test_acc: 0.3106
Epoch: 8 | train_loss: 2.8021 | train_acc: 0.4284 | test_loss: 3.0520 | test_acc: 0.3190
Epoch: 9 | train_loss: 2.7126 | train_acc: 0.4365 | test_loss: 2.9885 | test_acc: 0.3271
Epoch: 10 | train_loss: 2.6322 | train_acc: 0.4500 | test_loss: 2.9121 | test_acc: 0.3424
Epoch: 11 | train_loss: 2.5100 | train_acc: 0.4900 | test_loss: 2.8665 | test_acc: 0.3445
Epoch: 12 | train_l

{'train_loss': [1.2636193149494674],
 'train_acc': [0.7680817610062893],
 'test_loss': [2.2162507457553215],
 'test_acc': [0.4659198113207547]}

In [16]:
import torch
from tqdm.auto import tqdm

def train_with_scheduler(model: torch.nn.Module,
          writer,
          schedular: torch.optim.lr_scheduler,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader:  torch.utils.data.DataLoader,
          loss_fn: torch.nn.Module,
          optimizer: torch.optim.Optimizer,
          device,
          epochs: int = 10,
          save_model: bool = False,
          save_model_path: str = "./models",
          model_name: str = "vit_model"):
    
    for epoch in tqdm(range(epochs)):
        # Put the model in train mode
        model.to(device)
        model.train()

        # Setup train loss and train accuracy values
        train_loss, train_acc = 0, 0

        for batch, (X, y) in enumerate(train_dataloader):
            # Send data to target device
            X, y = X.to(device), y.to(device)

            # Forward pass
            y_pred = model(X)

            # Calculate and accumulate loss 
            loss = loss_fn(y_pred, y)
            train_loss += loss.item()

            # Optimizer zero grad
            optimizer.zero_grad()

            # Loss backward
            loss.backward()

            # Optimizer step
            optimizer.step()

            # Calculate and accumulate accuray metrics across all batches
            y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
            train_acc += (y_pred_class == y).sum().item()/len(y_pred)
       
        schedular.step()
        
        # Adjust metrics to get average loss and accuracy per batch
        train_loss = train_loss / len(train_dataloader)
        train_acc = train_acc / len(train_dataloader)

        # Put the model in eval mode
        model.to(device)
        model.eval()

        # Setup test loss and test accuracy values
        test_loss, test_acc = 0, 0

        with torch.inference_mode():
            # Loop through DataLoader batches
            for batch, (X, y) in enumerate(test_dataloader):
                # Send data to target deivce
                X, y = X.to(device), y.to(device)

                # Forward pass
                test_pred_logits = model(X)

                # Calculate and accumulate loss
                loss = loss_fn(test_pred_logits, y)
                test_loss += loss.item()

                # Calculate and accumulate accuracy
                test_pred_labels = test_pred_logits.argmax(dim=1) 
                test_acc += ((test_pred_labels == y).sum().item() / len(test_pred_labels))

            # Adjust metrics to get average loss and accuracy per batch
            test_loss = test_loss / len(train_dataloader)
            test_acc = test_acc / len(test_dataloader)
            
            results = {"train_loss": [],
                        "train_acc": [],
                        "test_loss": [],
                        "test_acc": []}
            print(
                    f"Epoch: {epoch+1} | "
                    f"train_loss: {train_loss:.4f} | "
                    f"train_acc: {train_acc:.4f} | "
                    f"test_loss: {test_loss:.4f} | "
                    f"test_acc: {test_acc:.4f}")
            results["train_loss"].append(train_loss)
            results["train_acc"].append(train_acc)
            results["test_loss"].append(test_loss)
            results["test_acc"].append(test_acc)

            writer.add_scalar(tag="Train Loss",
                              scalar_value=train_loss,
                              global_step=epoch)

            writer.add_scalar(tag="Train Accuracy",
                              scalar_value=train_acc,
                              global_step=epoch)             

            writer.add_scalar(tag="Test Loss",
                              scalar_value=test_loss,
                              global_step=epoch) 
            writer.add_scalar(tag="Test Accuracy",
                              scalar_value=test_acc,
                              global_step=epoch) 
            writer.add_graph(model=model,
                             input_to_model=torch.randn(64, 3, 224, 224).to(device))
    if save_model == True:
        print(f"[INFO] Saving {model_name} model to {save_model_path}")
        MODEL_PATH = Path(save_model_path) 
        MODEL_PATH.mkdir(parents=True,
                         exist_ok=True)
        MODEL_SAVE_PATH = MODEL_PATH/model_name
        torch.save(obj=model.state_dict(),
                   f=MODEL_SAVE_PATH)
        return results
    else:
        return results

In [79]:
from torch.utils.tensorboard import SummaryWriter
from torch.optim import lr_scheduler

vit_3_epochs_with_linear_schedular = create_vit(device=device,
                           num_classes=num_classes,
                           seed=42)

vit_3_epochs_with_linear_schedular.to(device)

loss_fn = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(params=vit_3_epochs_with_linear_schedular.parameters(),
                             lr=1e-3)

schedular = lr_scheduler.LinearLR(optimizer, start_factor=0.01, end_factor=0.0001, total_iters = 20, verbose=True) 

writer = SummaryWriter(log_dir=os.path.join("./results", "vit_3_epochs_with_linear_schedular"))

train_with_scheduler(model=vit_3_epochs_with_linear_schedular,
      train_dataloader=vit_train_dataloder,
      writer=writer,
      schedular=schedular,
      test_dataloader=vit_test_dataloder,
      loss_fn=loss_fn,
      optimizer=optimizer,
      device=device,
      epochs=3,
      save_model=False,
      save_model_path="./models",
      model_name="vit_3_epochs_with_linear_schedular")

Adjusting learning rate of group 0 to 1.0000e-05.


  0%|          | 0/3 [00:00<?, ?it/s]

Adjusting learning rate of group 0 to 9.5050e-06.
Epoch: 1 | train_loss: 4.6534 | train_acc: 0.0118 | test_loss: 4.6361 | test_acc: 0.0103


/opt/homebrew/Caskroom/miniconda/base/envs/AircraftDetection/lib/python3.10/site-packages/torch/__init__.py:1209: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert condition, message


Adjusting learning rate of group 0 to 9.0100e-06.
Epoch: 2 | train_loss: 4.6333 | train_acc: 0.0121 | test_loss: 4.6223 | test_acc: 0.0115
Adjusting learning rate of group 0 to 8.5150e-06.
Epoch: 3 | train_loss: 4.6205 | train_acc: 0.0149 | test_loss: 4.6067 | test_acc: 0.0115


{'train_loss': [4.620507051359932],
 'train_acc': [0.014937106918238994],
 'test_loss': [4.606733151201932],
 'test_acc': [0.011497641509433961]}

In [80]:
from torch.utils.tensorboard import SummaryWriter
from torch.optim import lr_scheduler

vit_3_epochs_with_expo_schedular = create_vit(device=device,
                           num_classes=num_classes,
                           seed=42)

vit_3_epochs_with_expo_schedular.to(device)

loss_fn = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(params=vit_3_epochs_with_expo_schedular.parameters(),
                             lr=1e-3)

schedular = lr_scheduler.ExponentialLR(optimizer, gamma=0.80) 

writer = SummaryWriter(log_dir=os.path.join("./results", "vit_3_epochs_with_expo_schedular"))

train_with_scheduler(model=vit_3_epochs_with_expo_schedular,
      train_dataloader=vit_train_dataloder,
      writer=writer,
      schedular=schedular,
      test_dataloader=vit_test_dataloder,
      loss_fn=loss_fn,
      optimizer=optimizer,
      device=device,
      epochs=3,
      save_model=True,
      save_model_path="./models",
      model_name="vit_3_epochs_with_expo_schedular")

  0%|          | 0/3 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 4.2990 | train_acc: 0.0580 | test_loss: 3.9562 | test_acc: 0.1143


/opt/homebrew/Caskroom/miniconda/base/envs/AircraftDetection/lib/python3.10/site-packages/torch/__init__.py:1209: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert condition, message


Epoch: 2 | train_loss: 3.8209 | train_acc: 0.1541 | test_loss: 3.7208 | test_acc: 0.1650
Epoch: 3 | train_loss: 3.6100 | train_acc: 0.2210 | test_loss: 3.5963 | test_acc: 0.1972
[INFO] Saving vit_3_epochs_with_expo_schedular model to ./models


{'train_loss': [3.609987924683769],
 'train_acc': [0.22101022012578614],
 'test_loss': [3.596314803609308],
 'test_acc': [0.1971698113207547]}

In [17]:
import torch
from tqdm.auto import tqdm

def train_with_reduceLROnPlateau_val_loss(model: torch.nn.Module,
          writer,
          schedular: torch.optim.lr_scheduler,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader:  torch.utils.data.DataLoader,
          loss_fn: torch.nn.Module,
          optimizer: torch.optim.Optimizer,
          device,
          epochs: int = 10,
          save_model: bool = False,
          save_model_path: str = "./models",
          model_name: str = "vit_model"):
    
    for epoch in tqdm(range(epochs)):
        # Put the model in train mode
        model.to(device)
        model.train()

        # Setup train loss and train accuracy values
        train_loss, train_acc = 0, 0

        for batch, (X, y) in enumerate(train_dataloader):
            # Send data to target device
            X, y = X.to(device), y.to(device)

            # Forward pass
            y_pred = model(X)

            # Calculate and accumulate loss 
            loss = loss_fn(y_pred, y)
            train_loss += loss.item()

            # Optimizer zero grad
            optimizer.zero_grad()

            # Loss backward
            loss.backward()

            # Optimizer step
            optimizer.step()

            # Calculate and accumulate accuray metrics across all batches
            y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
            train_acc += (y_pred_class == y).sum().item()/len(y_pred)
        
        # Adjust metrics to get average loss and accuracy per batch
        train_loss = train_loss / len(train_dataloader)
        train_acc = train_acc / len(train_dataloader)

        # Put the model in eval mode
        model.to(device)
        model.eval()

        # Setup test loss and test accuracy values
        test_loss, test_acc = 0, 0

        with torch.inference_mode():
            # Loop through DataLoader batches
            for batch, (X, y) in enumerate(test_dataloader):
                # Send data to target deivce
                X, y = X.to(device), y.to(device)

                # Forward pass
                test_pred_logits = model(X)

                # Calculate and accumulate loss
                loss = loss_fn(test_pred_logits, y)
                test_loss += loss.item()

                # Calculate and accumulate accuracy
                test_pred_labels = test_pred_logits.argmax(dim=1) 
                test_acc += ((test_pred_labels == y).sum().item() / len(test_pred_labels))

            # Adjust metrics to get average loss and accuracy per batch
            test_loss = test_loss / len(train_dataloader)
            test_acc = test_acc / len(test_dataloader)
            
            results = {"train_loss": [],
                        "train_acc": [],
                        "test_loss": [],
                        "test_acc": []}
            print(
                    f"Epoch: {epoch+1} | "
                    f"train_loss: {train_loss:.4f} | "
                    f"train_acc: {train_acc:.4f} | "
                    f"test_loss: {test_loss:.4f} | "
                    f"test_acc: {test_acc:.4f}")
            results["train_loss"].append(train_loss)
            results["train_acc"].append(train_acc)
            results["test_loss"].append(test_loss)
            results["test_acc"].append(test_acc)

            writer.add_scalar(tag="Train Loss",
                              scalar_value=train_loss,
                              global_step=epoch)

            writer.add_scalar(tag="Train Accuracy",
                              scalar_value=train_acc,
                              global_step=epoch)             

            writer.add_scalar(tag="Test Loss",
                              scalar_value=test_loss,
                              global_step=epoch) 
            writer.add_scalar(tag="Test Accuracy",
                              scalar_value=test_acc,
                              global_step=epoch) 
            writer.add_graph(model=model,
                             input_to_model=torch.randn(64, 3, 224, 224).to(device))
        print("Schedular Step")
        schedular.step(test_loss)

    if save_model == True:
        print(f"[INFO] Saving {model_name} model to {save_model_path}")
        MODEL_PATH = Path(save_model_path) 
        MODEL_PATH.mkdir(parents=True,
                         exist_ok=True)
        MODEL_SAVE_PATH = MODEL_PATH/model_name
        torch.save(obj=model.state_dict(),
                   f=MODEL_SAVE_PATH)
        return results
    else:
        return results

In [86]:
from torch.utils.tensorboard import SummaryWriter
from torch.optim import lr_scheduler

vit_3_epochs_with_reduceLROnPlateau = create_vit(device=device,
                           num_classes=num_classes,
                           seed=42)

vit_3_epochs_with_reduceLROnPlateau.to(device)

loss_fn = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(params=vit_3_epochs_with_reduceLROnPlateau.parameters(),
                             lr=1e-3)

schedular = lr_scheduler.ReduceLROnPlateau(optimizer, "max") 

writer = SummaryWriter(log_dir=os.path.join("./results", "vit_3_epochs_with_reduceLROnPlateau"))

train_with_reduceLROnPlateau_val_loss(model=vit_3_epochs_with_reduceLROnPlateau,
      train_dataloader=vit_train_dataloder,
      writer=writer,
      schedular=schedular,
      test_dataloader=vit_test_dataloder,
      loss_fn=loss_fn,
      optimizer=optimizer,
      device=device,
      epochs=3,
      save_model=False,
      save_model_path="./models",
      model_name="vit_3_epochs_with_reduceLROnPlateau")

  0%|          | 0/3 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 4.2990 | train_acc: 0.0580 | test_loss: 3.9562 | test_acc: 0.1143


/opt/homebrew/Caskroom/miniconda/base/envs/AircraftDetection/lib/python3.10/site-packages/torch/__init__.py:1209: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert condition, message


Schedular Step
Epoch: 2 | train_loss: 3.8070 | train_acc: 0.1569 | test_loss: 3.6965 | test_acc: 0.1608
Schedular Step
Epoch: 3 | train_loss: 3.5522 | train_acc: 0.2278 | test_loss: 3.5327 | test_acc: 0.1948
Schedular Step


{'train_loss': [3.552169808801615],
 'train_acc': [0.22779088050314464],
 'test_loss': [3.5327397832330667],
 'test_acc': [0.19481132075471697]}

In [18]:
from torch.utils.tensorboard import SummaryWriter
from torch.optim import lr_scheduler

vit_20_epochs_with_expo_schedular = create_vit(device=device,
                           num_classes=num_classes,
                           seed=42)

vit_20_epochs_with_expo_schedular.to(device)

loss_fn = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(params=vit_20_epochs_with_expo_schedular.parameters(),
                             lr=1e-3)

schedular = lr_scheduler.ExponentialLR(optimizer, gamma=0.80) 

writer = SummaryWriter(log_dir=os.path.join("./results", "vit_20_epochs_with_expo_schedular"))

train_with_scheduler(model=vit_20_epochs_with_expo_schedular,
      train_dataloader=vit_train_dataloder,
      writer=writer,
      schedular=schedular,
      test_dataloader=vit_test_dataloder,
      loss_fn=loss_fn,
      optimizer=optimizer,
      device=device,
      epochs=20,
      save_model=True,
      save_model_path="./models",
      model_name="vit_20_epochs_with_expo_schedular")

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 4.2990 | train_acc: 0.0580 | test_loss: 3.9562 | test_acc: 0.1143


/opt/homebrew/Caskroom/miniconda/base/envs/AircraftDetection/lib/python3.10/site-packages/torch/__init__.py:1209: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert condition, message


Epoch: 2 | train_loss: 3.8209 | train_acc: 0.1541 | test_loss: 3.7208 | test_acc: 0.1650
Epoch: 3 | train_loss: 3.6100 | train_acc: 0.2210 | test_loss: 3.5963 | test_acc: 0.1972
Epoch: 4 | train_loss: 3.4518 | train_acc: 0.2944 | test_loss: 3.4986 | test_acc: 0.2370
Epoch: 5 | train_loss: 3.3391 | train_acc: 0.3226 | test_loss: 3.4316 | test_acc: 0.2632
Epoch: 6 | train_loss: 3.2827 | train_acc: 0.3670 | test_loss: 3.3788 | test_acc: 0.2885
Epoch: 7 | train_loss: 3.2010 | train_acc: 0.3919 | test_loss: 3.3567 | test_acc: 0.2870
Epoch: 8 | train_loss: 3.1775 | train_acc: 0.3877 | test_loss: 3.3427 | test_acc: 0.2935
Epoch: 9 | train_loss: 3.1440 | train_acc: 0.4069 | test_loss: 3.3121 | test_acc: 0.2956
Epoch: 10 | train_loss: 3.1193 | train_acc: 0.4143 | test_loss: 3.2975 | test_acc: 0.2965
Epoch: 11 | train_loss: 3.0660 | train_acc: 0.4363 | test_loss: 3.2791 | test_acc: 0.3032
Epoch: 12 | train_loss: 3.0501 | train_acc: 0.4416 | test_loss: 3.2806 | test_acc: 0.2980
Epoch: 13 | train_

{'train_loss': [3.034171464308253],
 'train_acc': [0.44418238993710696],
 'test_loss': [3.229834439619532],
 'test_acc': [0.309375]}

### Traning 20 Epochs ViT Model with Data Augmentation

In [18]:
vit_train_transforms = torchvision.transforms.Compose([
    torchvision.transforms.RandomVerticalFlip(p=0.3),
    torchvision.transforms.RandomHorizontalFlip(p=0.3),
    torchvision.transforms.RandomRotation(degrees=15),
    torchvision.transforms.TrivialAugmentWide(),
    vit_train_transforms,
])

In [21]:
from torch.utils.tensorboard import SummaryWriter
from torch.optim import lr_scheduler

vit_20_epochs_with_expo_schedular_and_data_augmentation = create_vit(device=device,
                           num_classes=num_classes,
                           seed=42)

vit_20_epochs_with_expo_schedular_and_data_augmentation.to(device)

loss_fn = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(params=vit_20_epochs_with_expo_schedular_and_data_augmentation.parameters(),
                             lr=1e-3)

schedular = lr_scheduler.ExponentialLR(optimizer, gamma=0.80) 

writer = SummaryWriter(log_dir=os.path.join("./results", "vit_20_epochs_with_expo_schedular_and_data_augmentation"))

train_with_scheduler(model=vit_20_epochs_with_expo_schedular_and_data_augmentation,
      train_dataloader=vit_train_dataloder,
      writer=writer,
      schedular=schedular,
      test_dataloader=vit_test_dataloder,
      loss_fn=loss_fn,
      optimizer=optimizer,
      device=device,
      epochs=20,
      save_model=True,
      save_model_path="./models",
      model_name="vit_20_epochs_with_expo_schedular_and_data_augmentation")

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 4.2990 | train_acc: 0.0580 | test_loss: 3.9562 | test_acc: 0.1143


/opt/homebrew/Caskroom/miniconda/base/envs/AircraftDetection/lib/python3.10/site-packages/torch/__init__.py:1209: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert condition, message


Epoch: 2 | train_loss: 3.8209 | train_acc: 0.1541 | test_loss: 3.7208 | test_acc: 0.1650
Epoch: 3 | train_loss: 3.6100 | train_acc: 0.2210 | test_loss: 3.5963 | test_acc: 0.1972
Epoch: 4 | train_loss: 3.4518 | train_acc: 0.2944 | test_loss: 3.4986 | test_acc: 0.2370
Epoch: 5 | train_loss: 3.3391 | train_acc: 0.3226 | test_loss: 3.4316 | test_acc: 0.2632
Epoch: 6 | train_loss: 3.2827 | train_acc: 0.3670 | test_loss: 3.3788 | test_acc: 0.2885
Epoch: 7 | train_loss: 3.2010 | train_acc: 0.3919 | test_loss: 3.3567 | test_acc: 0.2870
Epoch: 8 | train_loss: 3.1775 | train_acc: 0.3877 | test_loss: 3.3427 | test_acc: 0.2935
Epoch: 9 | train_loss: 3.1440 | train_acc: 0.4069 | test_loss: 3.3121 | test_acc: 0.2956
Epoch: 10 | train_loss: 3.1193 | train_acc: 0.4143 | test_loss: 3.2975 | test_acc: 0.2965
Epoch: 11 | train_loss: 3.0660 | train_acc: 0.4363 | test_loss: 3.2791 | test_acc: 0.3032
Epoch: 12 | train_loss: 3.0501 | train_acc: 0.4416 | test_loss: 3.2806 | test_acc: 0.2980
Epoch: 13 | train_

{'train_loss': [3.034171464308253],
 'train_acc': [0.44418238993710696],
 'test_loss': [3.229834439619532],
 'test_acc': [0.309375]}

In [16]:
#TODO: Best Model so far

from torch.utils.tensorboard import SummaryWriter

vit_75_epochs = create_vit(device=device,
                           num_classes=num_classes,
                           seed=42)

vit_75_epochs.to(device)

seventy_five_epochs_loss_fn = torch.nn.CrossEntropyLoss()

seventy_five_epochs_optimizer = torch.optim.Adam(params=vit_75_epochs.parameters(),
                             lr=0.01)

writer = SummaryWriter(log_dir=os.path.join("./results", "75_try_two_epochs"))

train(model=vit_75_epochs,
      train_dataloader=vit_train_dataloder,
      writer=writer,
      test_dataloader=vit_test_dataloder,
      loss_fn=seventy_five_epochs_loss_fn,
      optimizer=seventy_five_epochs_optimizer,
      device=device,
      epochs=75,
      save_model=True,
      save_model_path="./models",
      model_name="vit_75_epochs")

  0%|          | 0/75 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 4.4277 | train_acc: 0.1182 | test_loss: 3.6920 | test_acc: 0.2051


/opt/homebrew/Caskroom/miniconda/base/envs/AircraftDetection/lib/python3.10/site-packages/torch/__init__.py:1209: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert condition, message


Epoch: 2 | train_loss: 3.1626 | train_acc: 0.2887 | test_loss: 3.1767 | test_acc: 0.2547
Epoch: 3 | train_loss: 2.6154 | train_acc: 0.4020 | test_loss: 2.9926 | test_acc: 0.3186
Epoch: 4 | train_loss: 2.2553 | train_acc: 0.4664 | test_loss: 2.8152 | test_acc: 0.3295
Epoch: 5 | train_loss: 2.0012 | train_acc: 0.5203 | test_loss: 2.7927 | test_acc: 0.3426
Epoch: 6 | train_loss: 1.9838 | train_acc: 0.5264 | test_loss: 2.6715 | test_acc: 0.3648
Epoch: 7 | train_loss: 1.7439 | train_acc: 0.5983 | test_loss: 2.6096 | test_acc: 0.3622
Epoch: 8 | train_loss: 1.7430 | train_acc: 0.5949 | test_loss: 2.6605 | test_acc: 0.3825
Epoch: 9 | train_loss: 1.7666 | train_acc: 0.5978 | test_loss: 2.6308 | test_acc: 0.3719
Epoch: 10 | train_loss: 1.6709 | train_acc: 0.6171 | test_loss: 2.6205 | test_acc: 0.3716
Epoch: 11 | train_loss: 1.5411 | train_acc: 0.6578 | test_loss: 2.6733 | test_acc: 0.3952
Epoch: 12 | train_loss: 1.4284 | train_acc: 0.6811 | test_loss: 2.6689 | test_acc: 0.3891
Epoch: 13 | train_

{'train_loss': [0.8101380518022573],
 'train_acc': [0.847877358490566],
 'test_loss': [3.0743822759052493],
 'test_acc': [0.4421580188679245]}

## Downloading augmented data

In [19]:
import torchvision
vit_train_transforms_data_augmentation = torchvision.transforms.Compose([
    torchvision.transforms.RandomVerticalFlip(p=0.3),
    torchvision.transforms.RandomHorizontalFlip(p=0.3),
    torchvision.transforms.RandomRotation(degrees=15),
    torchvision.transforms.TrivialAugmentWide(),
    vit_train_transforms,
])

In [20]:
vit_train_data_augmentation_path = Path("./vit_data_augmentation")
vit_train_data_data_augmentation = datasets.FGVCAircraft(root=vit_train_data_augmentation_path,
                                       split="train",
                                       transform=vit_train_transforms_data_augmentation,
                                       download=True)

vit_train_dataloder_data_augmentation = DataLoader(dataset=vit_train_data_data_augmentation,
                                 batch_size=BATCH_SIZE,
                                 shuffle=True,
                                 num_workers=NUM_WORKERS,
                                 pin_memory=True)

In [16]:
from torch.utils.tensorboard import SummaryWriter

vit_75_epochs = create_vit(device=device,
                           num_classes=num_classes,
                           seed=42)

vit_75_epochs.to(device)

seventy_five_epochs_loss_fn = torch.nn.CrossEntropyLoss()

seventy_five_epochs_optimizer = torch.optim.Adam(params=vit_75_epochs.parameters(),
                             lr=0.01)

writer = SummaryWriter(log_dir=os.path.join("./results", "75_epochs_with_data_augmentation"))

train(model=vit_75_epochs,
      train_dataloader=vit_train_dataloder_data_augmentation,
      writer=writer,
      test_dataloader=vit_test_dataloder,
      loss_fn=seventy_five_epochs_loss_fn,
      optimizer=seventy_five_epochs_optimizer,
      device=device,
      epochs=75,
      save_model=True,
      save_model_path="./models",
      model_name="vit_75_epochs_data_augmentation")

  0%|          | 0/75 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 4.7032 | train_acc: 0.0822 | test_loss: 3.8154 | test_acc: 0.1548


/opt/homebrew/Caskroom/miniconda/base/envs/AircraftDetection/lib/python3.10/site-packages/torch/__init__.py:1209: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert condition, message


Epoch: 2 | train_loss: 3.8016 | train_acc: 0.1893 | test_loss: 3.4286 | test_acc: 0.2237
Epoch: 3 | train_loss: 3.3450 | train_acc: 0.2528 | test_loss: 3.3436 | test_acc: 0.2364
Epoch: 4 | train_loss: 3.1159 | train_acc: 0.3068 | test_loss: 3.2551 | test_acc: 0.2284
Epoch: 5 | train_loss: 2.9768 | train_acc: 0.3435 | test_loss: 3.2395 | test_acc: 0.2642
Epoch: 6 | train_loss: 2.9211 | train_acc: 0.3457 | test_loss: 2.9321 | test_acc: 0.2903
Epoch: 7 | train_loss: 2.7906 | train_acc: 0.3835 | test_loss: 2.8551 | test_acc: 0.3036
Epoch: 8 | train_loss: 2.7912 | train_acc: 0.3741 | test_loss: 3.0712 | test_acc: 0.3012
Epoch: 9 | train_loss: 2.7499 | train_acc: 0.3908 | test_loss: 3.0409 | test_acc: 0.2962
Epoch: 10 | train_loss: 2.6070 | train_acc: 0.4245 | test_loss: 2.9492 | test_acc: 0.3233
Epoch: 11 | train_loss: 2.5533 | train_acc: 0.4339 | test_loss: 3.0053 | test_acc: 0.3192
Epoch: 12 | train_loss: 2.4242 | train_acc: 0.4567 | test_loss: 2.9066 | test_acc: 0.3354
Epoch: 13 | train_

{'train_loss': [2.2034375600095064],
 'train_acc': [0.5934551886792453],
 'test_loss': [3.0387069774123856],
 'test_acc': [0.3964622641509434]}

In [17]:
from torch.utils.tensorboard import SummaryWriter

vit_75_epochs = create_vit(device=device,
                           num_classes=num_classes,
                           seed=42)

vit_75_epochs.to(device)

seventy_five_epochs_loss_fn = torch.nn.CrossEntropyLoss()

seventy_five_epochs_optimizer = torch.optim.Adam(params=vit_75_epochs.parameters(),
                             lr=0.1)

writer = SummaryWriter(log_dir=os.path.join("./results", "75_epochs_with_data_augmentation_increased_lr"))

train(model=vit_75_epochs,
      train_dataloader=vit_train_dataloder_data_augmentation,
      writer=writer,
      test_dataloader=vit_test_dataloder,
      loss_fn=seventy_five_epochs_loss_fn,
      optimizer=seventy_five_epochs_optimizer,
      device=device,
      epochs=75,
      save_model=True,
      save_model_path="./models",
      model_name="vit_75_epochs_data_augmentation_increased_lr")

  0%|          | 0/75 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 28.6019 | train_acc: 0.0710 | test_loss: 15.4820 | test_acc: 0.1461
Epoch: 2 | train_loss: 18.6799 | train_acc: 0.1586 | test_loss: 15.0247 | test_acc: 0.1748
Epoch: 3 | train_loss: 16.6580 | train_acc: 0.2096 | test_loss: 16.4179 | test_acc: 0.1962
Epoch: 4 | train_loss: 17.5364 | train_acc: 0.2474 | test_loss: 15.0360 | test_acc: 0.2125
Epoch: 5 | train_loss: 16.9038 | train_acc: 0.2675 | test_loss: 17.4047 | test_acc: 0.2177
Epoch: 6 | train_loss: 17.6823 | train_acc: 0.2700 | test_loss: 15.8685 | test_acc: 0.2239
Epoch: 7 | train_loss: 17.7919 | train_acc: 0.2925 | test_loss: 14.6076 | test_acc: 0.2649
Epoch: 8 | train_loss: 18.1329 | train_acc: 0.2989 | test_loss: 17.6151 | test_acc: 0.2074
Epoch: 9 | train_loss: 18.8125 | train_acc: 0.3063 | test_loss: 16.4564 | test_acc: 0.2466
Epoch: 10 | train_loss: 17.3382 | train_acc: 0.3340 | test_loss: 16.7704 | test_acc: 0.2685
Epoch: 11 | train_loss: 16.8246 | train_acc: 0.3440 | test_loss: 16.4927 | test_acc: 0.26

{'train_loss': [18.006704114518076],
 'train_acc': [0.5298742138364779],
 'test_loss': [23.848975055622606],
 'test_acc': [0.3424528301886792]}

In [18]:
from torch.utils.data import ConcatDataset
vit_final_train_dataloader = ConcatDataset([vit_train_data_data_augmentation, vit_train_data]) 

vit_train_final_dataloader = DataLoader(dataset=vit_final_train_dataloader,
                                 batch_size=BATCH_SIZE,
                                 shuffle=True,
                                 num_workers=NUM_WORKERS,
                                 pin_memory=True)

In [19]:
from torch.utils.tensorboard import SummaryWriter

vit_75_epochs = create_vit(device=device,
                           num_classes=num_classes,
                           seed=42)

vit_75_epochs.to(device)

seventy_five_epochs_loss_fn = torch.nn.CrossEntropyLoss()

seventy_five_epochs_optimizer = torch.optim.Adam(params=vit_75_epochs.parameters(),
                             lr=0.01)

writer = SummaryWriter(log_dir=os.path.join("./results", "75_epochs_with_concat_dataset"))

train(model=vit_75_epochs,
      train_dataloader=vit_train_final_dataloader,
      writer=writer,
      test_dataloader=vit_test_dataloder,
      loss_fn=seventy_five_epochs_loss_fn,
      optimizer=seventy_five_epochs_optimizer,
      device=device,
      epochs=75,
      save_model=True,
      save_model_path="./models",
      model_name="75_epochs_with_concat_dataset")

  0%|          | 0/75 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 4.1344 | train_acc: 0.1532 | test_loss: 1.7049 | test_acc: 0.2130


/opt/homebrew/Caskroom/miniconda/base/envs/AircraftDetection/lib/python3.10/site-packages/torch/__init__.py:1209: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert condition, message


Epoch: 2 | train_loss: 2.9685 | train_acc: 0.3242 | test_loss: 1.4983 | test_acc: 0.2995
Epoch: 3 | train_loss: 2.6131 | train_acc: 0.4035 | test_loss: 1.4110 | test_acc: 0.3484
Epoch: 4 | train_loss: 2.3512 | train_acc: 0.4632 | test_loss: 1.4524 | test_acc: 0.3175
Epoch: 5 | train_loss: 2.1824 | train_acc: 0.5069 | test_loss: 1.4194 | test_acc: 0.3442
Epoch: 6 | train_loss: 2.1047 | train_acc: 0.5316 | test_loss: 1.3492 | test_acc: 0.3813
Epoch: 7 | train_loss: 2.0062 | train_acc: 0.5518 | test_loss: 1.3119 | test_acc: 0.3917
Epoch: 8 | train_loss: 1.9253 | train_acc: 0.5757 | test_loss: 1.3128 | test_acc: 0.4022
Epoch: 9 | train_loss: 1.9076 | train_acc: 0.5815 | test_loss: 1.3720 | test_acc: 0.3905
Epoch: 10 | train_loss: 1.8854 | train_acc: 0.5866 | test_loss: 1.3995 | test_acc: 0.3802
Epoch: 11 | train_loss: 1.8725 | train_acc: 0.6034 | test_loss: 1.3471 | test_acc: 0.3841
Epoch: 12 | train_loss: 1.7474 | train_acc: 0.6250 | test_loss: 1.3232 | test_acc: 0.4104
Epoch: 13 | train_

python(82517) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(82519) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(82522) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(82523) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(82524) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(82526) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(82527) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(82528) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(82529) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(82531) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(82715) Malloc

...

python(86315) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86317) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86318) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86320) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86321) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86322) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86326) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86338) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86345) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86370) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86534) Malloc

Epoch: 47 | train_loss: 1.6101 | train_acc: 0.7087 | test_loss: 1.5844 | test_acc: 0.4275


python(86669) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86671) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86673) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86676) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86684) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86685) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86686) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86690) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86692) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86694) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(86790) Malloc

Epoch: 48 | train_loss: 1.4760 | train_acc: 0.7227 | test_loss: 1.5779 | test_acc: 0.4373


python(87049) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(87053) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(87054) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(87055) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(87057) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(87059) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(87060) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(87061) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(87064) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(87065) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [21]:
import torchvision
vit_train_transforms_data_augmentation = torchvision.transforms.Compose([
    torchvision.transforms.RandomVerticalFlip(p=1),
    torchvision.transforms.RandomHorizontalFlip(p=0.8),
    torchvision.transforms.RandomRotation(degrees=15),
    torchvision.transforms.TrivialAugmentWide(),
    vit_train_transforms,
])

In [22]:
vit_train_data_augmentation_path = Path("./vit_data_augmentation_2")
vit_train_data_data_augmentation = datasets.FGVCAircraft(root=vit_train_data_augmentation_path,
                                       split="train",
                                       transform=vit_train_transforms_data_augmentation,
                                       download=True)

vit_train_dataloder_data_augmentation = DataLoader(dataset=vit_train_data_data_augmentation,
                                 batch_size=BATCH_SIZE,
                                 shuffle=True,
                                 num_workers=NUM_WORKERS,
                                 pin_memory=True)

100%|██████████| 2753340328/2753340328 [03:02<00:00, 15058384.72it/s]


Extracting vit_data_augmentation_2/fgvc-aircraft-2013b.tar.gz to vit_data_augmentation_2


In [ ]:
from torch.utils.tensorboard import SummaryWriter

vit_75_epochs = create_vit(device=device,
                           num_classes=num_classes,
                           seed=42)

vit_75_epochs.to(device)

seventy_five_epochs_loss_fn = torch.nn.CrossEntropyLoss()

seventy_five_epochs_optimizer = torch.optim.Adam(params=vit_75_epochs.parameters(),
                             lr=0.01)

writer = SummaryWriter(log_dir=os.path.join("./results", "75_epochs_with_concat_dataset_2"))

train(model=vit_75_epochs,
      train_dataloader=vit_train_dataloder_data_augmentation,
      writer=writer,
      test_dataloader=vit_test_dataloder,
      loss_fn=seventy_five_epochs_loss_fn,
      optimizer=seventy_five_epochs_optimizer,
      device=device,
      epochs=75,
      save_model=True,
      save_model_path="./models",
      model_name="75_epochs_with_concat_dataset_2")

In [30]:
image, label = iter(next(vit_train_dataloder_data_augmentation))
image, label

TypeError: 'DataLoader' object is not an iterator

In [38]:
def create_resnet18(device,
                  num_classes: int = 100,
                  seed: int = 42):
    # Get the model and weights
    weights = torchvision.models.ResNet18_Weights.DEFAULT
    model = torchvision.models.resnet18(weights=weights).to(device)

    # Freeze feature extraction layers
    for param in model.parameters():
        param.requires_grad = False

    torch.manual_seed(seed)
    torch.mps.manual_seed(seed)

    # Create a new classifier layer
    model.fc = nn.Sequential(nn.Linear(in_features=512, out_features=num_classes))

    return model